<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
《<a href="https://mng.bz/lZ5B">Build a Reasoning Model (From Scratch)</a>》配套补充代码，作者 <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>代码仓库：<a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>



# 附录C：Qwen3 LLM 源代码


本 Notebook 使用到的依赖：


In [ ]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",  # 用于下载函数
    "torch",
    "tokenizers"
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

- 虽然本书强调“from scratch”，但正如前文所述，这里的 from scratch 专指推理方法，而非 LLM 本身。
- 真正从零实现一个 LLM 足以写成另一本书——这正是我另一本书 [Build A Large Language Model (From Scratch)](https://github.com/rasbt/LLMs-from-scratch) 的主题。
- 不过如果你想了解本书所用实现，本附录列出了我们通过 reasoning_from_scratch Python 包导入的 Qwen3 源码：

```python
from reasoning_from_scratch.qwen3 import Qwen3Model, Qwen3Tokenizer
```

- 请注意，这套架构代码与 [Build A Large Language Model (From Scratch)](https://github.com/rasbt/LLMs-from-scratch) 中介绍的 GPT-2 十分相似。
- 虽然本书不要求熟悉 GPT-2，但附录中仍加入了一些与 GPT-2 的对比，方便了解两者关系（这份实现正是我把 GPT-2 代码逐步迁移到 Qwen3 架构的结果）。


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-c/Appendix_C_F01_raschka.webp" width="500px">

&nbsp;
## C.1 均方根层归一化（RMSNorm）


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-c/Appendix_C_F02_raschka.webp" width="500px">

In [ ]:
import torch.nn as nn


class RMSNorm(nn.Module):
    def __init__(
        self,
        emb_dim,
        eps=1e-6,
        bias=False,
        qwen3_compatible=True,
    ):
        super().__init__()
        self.eps = eps
        self.qwen3_compatible = qwen3_compatible
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim)) if bias else None

    def forward(self, x):
        input_dtype = x.dtype

        if self.qwen3_compatible:
            x = x.to(torch.float32)

        variance = x.pow(2).mean(dim=-1, keepdim=True)
        norm_x = x * torch.rsqrt(variance + self.eps)
        norm_x = norm_x * self.scale

        if self.shift is not None:
            norm_x = norm_x + self.shift

        return norm_x.to(input_dtype)

&nbsp;
## C.2 前馈模块


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-c/Appendix_C_F03_raschka.webp" width="500px">

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc1 = nn.Linear(
            cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"],
            bias=False
        )
        self.fc2 = nn.Linear(
            cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"],
            bias=False
        )
        self.fc3 = nn.Linear(
            cfg["hidden_dim"], cfg["emb_dim"], dtype=cfg["dtype"],
            bias=False
        )

    def forward(self, x):
        x_fc1 = self.fc1(x)
        x_fc2 = self.fc2(x)
        x = nn.functional.silu(x_fc1) * x_fc2
        return self.fc3(x)

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-c/Appendix_C_F04_raschka.webp" width="500px">

&nbsp;
## C.3 旋转位置嵌入（RoPE）


In [ ]:
import torch


def compute_rope_params(head_dim, theta_base=10_000, context_length=4096,
                        dtype=torch.float32):
    assert head_dim % 2 == 0, "Embedding dimension must be even"
    inv_freq = 1.0 / (theta_base ** (
        torch.arange(0, head_dim, 2, dtype=dtype)[: (head_dim // 2)].float()
        / head_dim
    ))
    positions = torch.arange(context_length, dtype=dtype)
    angles = positions[:, None] * inv_freq[None, :]
    angles = torch.cat([angles, angles], dim=1)

    cos = torch.cos(angles)
    sin = torch.sin(angles)

    return cos, sin


def apply_rope(x, cos, sin, offset=0):
    # x 的形状：(batch_size, num_heads, seq_len, head_dim)
    batch_size, num_heads, seq_len, head_dim = x.shape
    assert head_dim % 2 == 0, "Head dimension must be even"

    # 将 x 拆分为前半部分和后半部分
    x1 = x[..., : head_dim // 2]  # 前半部分
    x2 = x[..., head_dim // 2:]  # 后半部分

    # 调整 sin 和 cos 的形状，使其为 (1, 1, seq_len, head_dim)
    cos = cos[offset:offset + seq_len, :].unsqueeze(0).unsqueeze(0)  
    sin = sin[offset:offset + seq_len, :].unsqueeze(0).unsqueeze(0)

    rotated = torch.cat((-x2, x1), dim=-1)
    x_rotated = (x * cos) + (rotated * sin)

    # 应用 cos 和 sin 旋转后可以使用较低精度
    return x_rotated.to(dtype=x.dtype)

&nbsp;
## C.4 分组查询注意力（GQA）


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-c/Appendix_C_F05_raschka.webp" width="500px">

In [ ]:
class GroupedQueryAttention(nn.Module):
    def __init__(self, d_in, num_heads, num_kv_groups, head_dim=None,
                 qk_norm=False, dtype=None):
        super().__init__()
        assert num_heads % num_kv_groups == 0

        self.num_heads = num_heads
        self.num_kv_groups = num_kv_groups
        self.group_size = num_heads // num_kv_groups

        if head_dim is None:
            assert d_in % num_heads == 0
            head_dim = d_in // num_heads

        self.head_dim = head_dim
        self.d_out = num_heads * head_dim

        self.W_query = nn.Linear(
            d_in, self.d_out, bias=False, dtype=dtype
        )
        self.W_key = nn.Linear(
            d_in, num_kv_groups * head_dim, bias=False,dtype=dtype
        )
        self.W_value = nn.Linear(
            d_in, num_kv_groups * head_dim, bias=False, dtype=dtype
        )

        self.out_proj = nn.Linear(self.d_out, d_in, bias=False, dtype=dtype)

        if qk_norm:
            self.q_norm = RMSNorm(head_dim, eps=1e-6)
            self.k_norm = RMSNorm(head_dim, eps=1e-6)
        else:
            self.q_norm = self.k_norm = None

    def forward(self, x, mask, cos, sin, start_pos=0, cache=None):
        b, num_tokens, _ = x.shape

        queries = self.W_query(x)  # 形状：(b, num_tokens, num_heads * head_dim)
        keys = self.W_key(x)       # 形状：(b, num_tokens, num_kv_groups * head_dim)
        values = self.W_value(x)   # 形状：(b, num_tokens, num_kv_groups * head_dim)

        queries = queries.view(b, num_tokens, self.num_heads,
                               self.head_dim).transpose(1, 2)
        keys_new = keys.view(b, num_tokens, self.num_kv_groups,
                             self.head_dim).transpose(1, 2)
        values_new = values.view(b, num_tokens, self.num_kv_groups,
                                 self.head_dim).transpose(1, 2)

        if self.q_norm:
            queries = self.q_norm(queries)
        if self.k_norm:
            keys_new = self.k_norm(keys_new)

        queries = apply_rope(queries, cos, sin, offset=start_pos)
        keys_new = apply_rope(keys_new, cos, sin, offset=start_pos)

        if cache is not None:
            prev_k, prev_v = cache
            keys = torch.cat([prev_k, keys_new], dim=2)
            values = torch.cat([prev_v, values_new], dim=2)
        else:
            start_pos = 0  # 重置 RoPE
            keys, values = keys_new, values_new
        next_cache = (keys, values)

        # 扩展 K 和 V 以匹配注意力头数量
        keys = keys.repeat_interleave(self.group_size, dim=1)
        values = values.repeat_interleave(self.group_size, dim=1)

        attn_scores = queries @ keys.transpose(2, 3)
        attn_scores = attn_scores.masked_fill(mask, -torch.inf)
        attn_weights = torch.softmax(
            attn_scores / self.head_dim**0.5, dim=-1
        )

        context = (attn_weights @ values).transpose(1, 2)
        context = context.reshape(b, num_tokens, self.d_out)
        return self.out_proj(context), next_cache

&nbsp;
## C.5 Transformer 块


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-c/Appendix_C_F06_raschka.webp" width="200px">

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = GroupedQueryAttention(
            d_in=cfg["emb_dim"],
            num_heads=cfg["n_heads"],
            head_dim=cfg["head_dim"],
            num_kv_groups=cfg["n_kv_groups"],
            qk_norm=cfg["qk_norm"],
            dtype=cfg["dtype"]
        )
        self.ff = FeedForward(cfg)
        self.norm1 = RMSNorm(cfg["emb_dim"], eps=1e-6)
        self.norm2 = RMSNorm(cfg["emb_dim"], eps=1e-6)

    def forward(self, x, mask, cos, sin, start_pos=0, cache=None):
        # 注意力块的残差连接
        shortcut = x
        x = self.norm1(x)
        x, next_cache = self.att(
            x, mask, cos, sin, start_pos=start_pos,cache=cache
        )  # 形状为 [batch_size, num_tokens, emb_size]
        x = x + shortcut  # 加回原始输入

        # 前馈网络块的残差连接
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = x + shortcut  # 加回原始输入

        return x, next_cache

&nbsp;
## C.6 主模型代码


In [ ]:
class Qwen3Model(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        # 主要模型参数
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"],
                                    dtype=cfg["dtype"])

        self.trf_blocks = nn.ModuleList(
            [TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )
        self.final_norm = RMSNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"],
            bias=False, dtype=cfg["dtype"]
        )

        # 可复用的辅助组件
        if cfg["head_dim"] is None:
            head_dim = cfg["emb_dim"] // cfg["n_heads"]
        else:
            head_dim = cfg["head_dim"]
        cos, sin = compute_rope_params(
            head_dim=head_dim,
            theta_base=cfg["rope_base"],
            context_length=cfg["context_length"]
        )
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)
        self.cfg = cfg
        self.current_pos = 0  # 记录 KV 缓存中的当前位置

    def forward(self, in_idx, cache=None):
        # 前向传播
        tok_embeds = self.tok_emb(in_idx)
        x = tok_embeds

        num_tokens = x.shape[1]
        if cache is not None:
            pos_start = self.current_pos
            pos_end = pos_start + num_tokens
            self.current_pos = pos_end
            mask = torch.triu(
                torch.ones(
                    pos_end, pos_end, device=x.device, dtype=torch.bool
                ),
                diagonal=1
            )[pos_start:pos_end, :pos_end]
        else:
            pos_start = 0  # 并非必需，但有助于 torch.compile
            mask = torch.triu(
                torch.ones(num_tokens, num_tokens, device=x.device,
                           dtype=torch.bool),
                diagonal=1
            )
        # 预填充（无缓存）时，掩码初始形状为 (num_tokens, num_tokens)
        # 缓存解码时，掩码初始形状为 (num_tokens, prev_k_number_tokens + num_tokens)
        #
        # 添加两个前导维度，使掩码变为
        # 预填充时的 (1, 1, num_tokens, num_tokens)，以及
        # 缓存解码时的 (1, 1, num_tokens, total_key_tokens)。
        # 这些额外维度使 PyTorch 能够广播同一个掩码
        # 并在将其应用于以下形状的 attn_scores 时覆盖所有批次和注意力头：
        # 形状：(batch, num_heads, num_tokens, total_key_tokens)。
        mask = mask[None, None, :, :]

        for i, block in enumerate(self.trf_blocks):
            blk_cache = cache.get(i) if cache else None
            x, new_blk_cache = block(x, mask, self.cos, self.sin,
                                     start_pos=pos_start,
                                     cache=blk_cache)
            if cache is not None:
                cache.update(i, new_blk_cache)

        x = self.final_norm(x)
        logits = self.out_head(x.to(self.cfg["dtype"]))
        return logits

    def reset_kv_cache(self):
        self.current_pos = 0

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-c/Appendix_C_F07_raschka.webp" width="350px">

In [ ]:
QWEN_CONFIG_06_B = {
    "vocab_size": 151_936,     # 词表大小
    "context_length": 40_960,  # 模型训练时使用的上下文长度
    "emb_dim": 1024,           # 嵌入维度
    "n_heads": 16,             # 注意力头数量
    "n_layers": 28,            # 层数
    "hidden_dim": 3072,        # 前馈网络中间维度的大小
    "head_dim": 128,           # GQA 中每个头的维度
    "qk_norm": True,           # 是否对 GQA 中的查询和键进行归一化
    "n_kv_groups": 8,          # 分组查询注意力中的键值组数量
    "rope_base": 1_000_000.0,  # RoPE 中 theta 的基数
    "dtype": torch.bfloat16,   # 使用较低精度的数据类型以减少内存占用
}

- Qwen3 0.6B 模型的总上下文长度支持 40,960 个 token。
- 在这 40,960 个 token 中，约 32,768 个用于模型输出（生成文本），8,192 个用于常见的提示（用户输入）。
- 模型训练时使用 32,768 token 的序列，因此该长度最能反映模型的有效工作范围。
- 32,768 个 token 已经绰绰有余；举个例子，约 4 万 token 的上下文相当于《哈利·波特》第一部篇幅的一半。


&nbsp;
## C.7 KV 缓存


In [ ]:
class KVCache:
    def __init__(self, n_layers):
        self.cache = [None] * n_layers

    def get(self, layer_idx):
        return self.cache[layer_idx]

    def update(self, layer_idx, value):
        self.cache[layer_idx] = value

    def get_all(self):
        return self.cache

    def reset(self):
        for i in range(len(self.cache)):
            self.cache[i] = None

&nbsp;
## C.8 分词器


In [ ]:
import re
from tokenizers import Tokenizer

class Qwen3Tokenizer:
    _SPECIALS = [
        "<|endoftext|>",
        "<|im_start|>", "<|im_end|>",
        "<|object_ref_start|>", "<|object_ref_end|>",
        "<|box_start|>", "<|box_end|>",
        "<|quad_start|>", "<|quad_end|>",
        "<|vision_start|>", "<|vision_end|>",
        "<|vision_pad|>", "<|image_pad|>", "<|video_pad|>",
    ]
    _SPLIT_RE = re.compile(r"(<\|[^>]+?\|>)")

    def __init__(self, tokenizer_file_path="tokenizer-base.json",
                 apply_chat_template=False,
                 add_generation_prompt=False,
                 add_thinking=False):

        self.apply_chat_template = apply_chat_template
        self.add_generation_prompt = add_generation_prompt
        self.add_thinking = add_thinking

        tok_path = Path(tokenizer_file_path)
        if not tok_path.is_file():
            raise FileNotFoundError(
                f"Tokenizer file '{tok_path}' not found. "
            )

        self._tok = Tokenizer.from_file(str(tok_path))
        self._special_to_id = {t: self._tok.token_to_id(t) 
                               for t in self._SPECIALS}

        self.pad_token = "<|endoftext|>"
        self.pad_token_id = self._special_to_id.get(self.pad_token)

        # 与 HF 行为保持一致：对话模型使用 <|im_end|>，基础模型使用 <|endoftext|>
        fname = tok_path.name.lower()
        if "base" in fname and "reasoning" not in fname:
            self.eos_token = "<|endoftext|>"
        else:
            self.eos_token = "<|im_end|>"
        self.eos_token_id = self._special_to_id.get(self.eos_token)

    def encode(self, prompt, chat_wrapped=None):
        if chat_wrapped is None:
            chat_wrapped = self.apply_chat_template

        stripped = prompt.strip()
        if stripped in self._special_to_id and "\n" not in stripped:
            return [self._special_to_id[stripped]]

        if chat_wrapped:
            prompt = self._wrap_chat(prompt)

        ids = []
        for part in filter(None, self._SPLIT_RE.split(prompt)):
            if part in self._special_to_id:
                ids.append(self._special_to_id[part])
            else:
                ids.extend(self._tok.encode(part).ids)
        return ids

    def decode(self, token_ids):
        return self._tok.decode(token_ids, skip_special_tokens=False)

    def _wrap_chat(self, user_msg):
        s = f"<|im_start|>user\n{user_msg}<|im_end|>\n"
        if self.add_generation_prompt:
            s += "<|im_start|>assistant"
            if self.add_thinking:
                s += "\n"  # 不插入 <think> 标签，只添加一个换行符
            else:
                s += "\n<think>\n\n</think>\n\n"
        return s

&nbsp;
## C.9 使用模型


In [ ]:
from pathlib import Path
import torch

from reasoning_from_scratch.ch02 import get_device  # noqa: F401
from reasoning_from_scratch.qwen3 import download_qwen3_small

# device = get_device()  # 可选：取消注释以使用自动设备选择器
device = torch.device("cpu")

download_qwen3_small(kind="base", tokenizer_only=False, out_dir="qwen3")

tokenizer_file_path = Path("qwen3") / "tokenizer-base.json"
model_file = Path("qwen3") / "qwen3-0.6B-base.pth"

tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_file_path)
model = Qwen3Model(QWEN_CONFIG_06_B)
model.load_state_dict(torch.load(model_file))

model.to(device)

In [ ]:
import time
from reasoning_from_scratch.ch02 import (
   generate_text_basic_stream_cache,
   generate_stats
)


prompt = "Explain large language models in a single sentence."
input_token_ids_tensor = torch.tensor(
    tokenizer.encode(prompt),
    device=device
    ).unsqueeze(0)
max_new_tokens = 200

start_time = time.time()
generated_ids = []

for token in generate_text_basic_stream_cache(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
    eos_token_id=tokenizer.eos_token_id
):
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
    )

    next_token_id = token.squeeze(0)
    generated_ids.append(next_token_id)  # 收集生成的词元

end_time = time.time()

output_token_ids_tensor = torch.cat(generated_ids, dim=0)
generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)